# ROS, comet assay, and RT-qPCR

This notebook links the three molecular-response datasets to the manuscript figures. Treatment labels are standardized as SCB and SCF. Data-quality decisions are documented in `data/processed/MOLECULAR_DATA_NOTES.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ARTICLE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA = ARTICLE_DIR / 'data' / 'processed'
FIGURES = ARTICLE_DIR / 'results' / 'figures'
MANUSCRIPT = ARTICLE_DIR / 'artigo_andre_tcc' / 'images'
FIGURES.mkdir(parents=True, exist_ok=True)
MANUSCRIPT.mkdir(parents=True, exist_ok=True)
LABELS = {'SCB': 'SCB', 'SCF': 'SCF', 'control': 'Control', 'Control': 'Control',
          'control_negative': 'Negative control', 'control_positive': 'Positive control'}

def export(fig, stem):
    for folder in [FIGURES, MANUSCRIPT]:
        fig.savefig(folder / f'{stem}.pdf', bbox_inches='tight')
        fig.savefig(folder / f'{stem}.png', dpi=600, bbox_inches='tight')

## ROS

The legacy monograph used a Kruskal–Wallis workflow and manually annotated significance. The final annotations must come from stored test results. `Snap-19` is intentionally left missing until its adjacent value is verified against the original export.

In [ ]:
ros = pd.read_excel(DATA / 'ros_intensity_22082025.xlsx')
ros['treatment'] = ros['type'].replace(LABELS)
ros_plot = ros.dropna(subset=['intensity', 'treatment', 'concentration']).copy()
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.boxplot(data=ros_plot, x='concentration', y='intensity', hue='treatment', showfliers=False, ax=ax)
sns.stripplot(data=ros_plot, x='concentration', y='intensity', hue='treatment', dodge=True, color='black', alpha=.55, legend=False, ax=ax)
ax.set(xlabel='Concentration (CB/L)', ylabel='Fluorescence intensity (a.u.)')
sns.despine(); fig.tight_layout(); export(fig, 'ros_intensity')

## Comet assay

The main panel excludes positive and negative controls, matching the monograph. The SI panel retains controls and outliers. The cleaned dataset lacks stable cell/image identifiers, which limits traceability back to individual microscopy files.

In [ ]:
comet = pd.read_excel(DATA / 'comet_assay_clean.xlsx')
comet['treatment'] = comet['type'].replace(LABELS)
main = comet.loc[~comet['type'].isin(['control_negative', 'control_positive'])].copy()
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.boxplot(data=main, x='concentration', y='tail_moment', hue='treatment', showfliers=False, ax=ax)
sns.stripplot(data=main, x='concentration', y='tail_moment', hue='treatment', dodge=True, color='black', alpha=.5, legend=False, ax=ax)
ax.set(xlabel='Concentration (CB/L)', ylabel='Olive tail moment')
sns.despine(); fig.tight_layout(); export(fig, 'comet_assay')

## RT-qPCR

The source columns `il1` and `tnf` are renamed to `il1b` and `tnfa`. The selected targets came from separate NLP extractions for nicotine, heavy metals, and PAHs, followed by primer-availability screening. Confirm whether rows are technical rather than biological replicates before reporting inferential p-values.

In [ ]:
qpcr = pd.read_excel(DATA / 'rt_qpcr_expression.xlsx').rename(columns={'il1': 'il1b', 'tnf': 'tnfa'})
qpcr['treatment'] = qpcr['type'].replace(LABELS)
genes = ['nestin', 'gfap', 'il1b', 'tnfa', 'casp9']
long = qpcr.melt(id_vars=['treatment', 'concentration'], value_vars=genes,
                 var_name='gene', value_name='relative_expression').dropna()
fig, ax = plt.subplots(figsize=(7.2, 4.2))
sns.barplot(data=long, x='gene', y='relative_expression', hue='treatment', errorbar='sd', capsize=.08, ax=ax)
sns.stripplot(data=long, x='gene', y='relative_expression', hue='treatment', dodge=True, color='black', alpha=.55, legend=False, ax=ax)
ax.axhline(1, color='0.5', lw=1, ls='--')
ax.set(xlabel='', ylabel='Relative expression')
sns.despine(); fig.tight_layout(); export(fig, 'rt_qpcr_expression')